In [1]:
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
%matplotlib widget

from spectra.utilities import read_OE65PRO_spectrum_data_file, find_index_of_nearest

In [2]:
lightsource_spectrometer_file = Path('raw_data') / 'QEPB00791_17-07-21-669.txt'

lightsource_spectrometer_header, lightsource_spectrometer_data = read_OE65PRO_spectrum_data_file(lightsource_spectrometer_file)
lightsource_spectrometer_data.shape

raw_data/QEPB00791_17-07-21-669.txt


(1044, 2)

In [3]:
PEGDA_spectrometer_file = Path('raw_data') / 'QEPB00791_17-10-07-458.txt'

PEGDA_spectrometer_header, PEGDA_spectrometer_data = read_OE65PRO_spectrum_data_file(PEGDA_spectrometer_file)
PEGDA_spectrometer_data.shape

raw_data/QEPB00791_17-10-07-458.txt


(1044, 2)

In [4]:
avobenzone_spectrometer_file = Path('raw_data') / 'QEPB00791_17-18-57-140.txt'

avobenzone_spectrometer_header, avobenzone_spectrometer_data = read_OE65PRO_spectrum_data_file(avobenzone_spectrometer_file)
avobenzone_spectrometer_data.shape

raw_data/QEPB00791_17-18-57-140.txt


(1044, 2)

In [5]:
fig, ax = plt.subplots(figsize=(10,5))
ax.plot(PEGDA_spectrometer_data[:,0], PEGDA_spectrometer_data[:,1], label='PEGDA')
ax.plot(avobenzone_spectrometer_data[:,0], avobenzone_spectrometer_data[:,1], label='avobenzone')
# ax.plot(lightsource_spectrometer_data[:,0], lightsource_spectrometer_data[:,1], label='Light Source')
ax.set_xlabel('Wavelength (nm)')
ax.set_ylabel('Signal')
ax.set_title("Raw Spectrum")
# ax.set_xlim([320, 460])
# ax.set_ylim([0, 60000])
ax.grid()
ax.legend();

Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …

# Restrict data sets to 300-540 nm

Reason:
- Low end - arbitrary
- High end - just between 532 nm laser wavelength and saturated spectrum values.

In [6]:
# Make sure both data sets are for the same wavelengths
assert np.allclose(PEGDA_spectrometer_data[:,0], avobenzone_spectrometer_data[:,0])

In [16]:
wavelength_range = [300, 540]

index_low = find_index_of_nearest(PEGDA_spectrometer_data[:,0], wavelength_range[0])
index_high = find_index_of_nearest(PEGDA_spectrometer_data[:,0], wavelength_range[1])

print(index_low, PEGDA_spectrometer_data[index_low, :])
print(index_high, PEGDA_spectrometer_data[index_high, :])

126 [299.969 222.   ]
431 [  539.971 25118.   ]


# Calculate absorbance

In [17]:
temp = np.copy(PEGDA_spectrometer_data[index_low:index_high + 1, :])
temp[:,1] = avobenzone_spectrometer_data[index_low:index_high + 1,1] / temp[:,1]
absorbance = np.column_stack((temp[:,0], -np.log10(temp[:,1])))

absorbance.shape

(306, 2)

In [19]:
fig, ax = plt.subplots()
ax.plot(absorbance[:,0], absorbance[:,1])
ax.set_xlabel('Wavelength (nm)')
ax.set_ylabel('Signal')
ax.set_title("Avobenzone Absorbance")
ax.grid();

Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …